In [1]:
# ====================== Colab: Data Preparation ======================
!pip install -q datasets

from datasets import load_dataset
import json
import os

# GSM8K load
dataset = load_dataset("openai/gsm8k", "main")

def format_example(example):
    question = example["question"]
    answer = example["answer"]

    # Chain-of-Thought format
    prompt = f"""Solve the following math problem step by step. Put the final answer after the step-by-step solution.

Problem: {question}

Solution:"""

    return {
        "prompt": prompt,
        "completion": answer,
        "question": question,
        "answer": answer
    }

train_data = dataset["train"].map(format_example)
test_data = dataset["test"].map(format_example)

# সেভ করো
os.makedirs("/content/gsm8k_processed", exist_ok=True)

train_data.save_to_disk("/content/gsm8k_processed/train")
test_data.save_to_disk("/content/gsm8k_processed/test")

# Zip করে ডাউনলোড
!zip -r gsm8k_processed.zip /content/gsm8k_processed
from google.colab import files
files.download("gsm8k_processed.zip")

print("Zip download")

README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.31MB            

main/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

main/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  419kB            

main/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

... (5 outputs left out)

  adding: content/gsm8k_processed/ (stored 0%)
  adding: content/gsm8k_processed/test/ (stored 0%)
  adding: content/gsm8k_processed/test/data-00000-of-00001.arrow (deflated 67%)
  adding: content/gsm8k_processed/test/state.json (deflated 37%)
  adding: content/gsm8k_processed/test/dataset_info.json (deflated 67%)
  adding: content/gsm8k_processed/train/ (stored 0%)
  adding: content/gsm8k_processed/train/data-00000-of-00001.arrow (deflated 67%)
  adding: content/gsm8k_processed/train/state.json (deflated 38%)
  adding: content/gsm8k_processed/train/dataset_info.json (deflated 67%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

ডেটা প্রস্তুত! zip ফাইল ডাউনলোড হয়েছে।


In [16]:
%%capture
!pip install unsloth
!pip install --upgrade --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

In [18]:
!pip uninstall -y transformers peft accelerate bitsandbytes trl unsloth unsloth_zoo -q
!pip install -q transformers==4.46.3 peft==0.13.2 accelerate==1.1.0 bitsandbytes datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 63.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.2/333.2 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 81.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [19]:
from datasets import load_dataset

dataset = load_dataset("openai/gsm8k", "main", split="train")

def format_example(example):
    text = f"""Solve the following math problem step by step. Put the final answer after the step-by-step solution.

Problem: {example['question']}

Solution: {example['answer']}"""
    return {"text": text}

dataset = dataset.map(format_example)
print(dataset[0]["text"][:250])
print("\nTotal samples:", len(dataset))

Map:   0%|          | 0/7473 [00:00<?, ? examples/s]

Solve the following math problem step by step. Put the final answer after the step-by-step solution.

Problem: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether i

Total samples: 7473


In [20]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print("Model loaded successfully!")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820
Model loaded successfully!


In [21]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=1024,
        padding="max_length",
    )

tokenized_dataset = dataset.map(tokenize, remove_columns=dataset.column_names)

training_args = TrainingArguments(
    output_dir="./qwen-1.5b-gsm8k-lora",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=2,
    logging_steps=10,
    save_strategy="epoch",
    fp16=True,
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    warmup_steps=50,
    report_to="none",
    remove_unused_columns=False,
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

print("Training...")
trainer.train()

trainer.save_model("./qwen-1.5b-gsm8k-lora")
tokenizer.save_pretrained("./qwen-1.5b-gsm8k-lora")
print("\n✅ Training complete!")

Map:   0%|          | 0/7473 [00:00<?, ? examples/s]

Training শুরু হচ্ছে...


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,0.938578
20,0.728840
30,0.507324
40,0.463821
50,0.460406
60,0.459895
70,0.402903
80,0.429083
90,0.435827
100,0.410009


Step,Training Loss
10,0.938578
20,0.728840
30,0.507324
40,0.463821
50,0.460406
60,0.459895
70,0.402903
80,0.429083
90,0.435827
100,0.410009


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)



✅ Training complete!


In [22]:
!zip -r qwen-1.5b-gsm8k-lora.zip qwen-1.5b-gsm8k-lora
from google.colab import files
files.download("qwen-1.5b-gsm8k-lora.zip")

  adding: qwen-1.5b-gsm8k-lora/ (stored 0%)
  adding: qwen-1.5b-gsm8k-lora/adapter_model.safetensors (deflated 7%)
  adding: qwen-1.5b-gsm8k-lora/tokenizer_config.json (deflated 60%)
  adding: qwen-1.5b-gsm8k-lora/training_args.bin (deflated 53%)
  adding: qwen-1.5b-gsm8k-lora/checkpoint-468/ (stored 0%)
  adding: qwen-1.5b-gsm8k-lora/checkpoint-468/trainer_state.json (deflated 73%)
  adding: qwen-1.5b-gsm8k-lora/checkpoint-468/adapter_model.safetensors (deflated 7%)
  adding: qwen-1.5b-gsm8k-lora/checkpoint-468/tokenizer_config.json (deflated 60%)
  adding: qwen-1.5b-gsm8k-lora/checkpoint-468/training_args.bin (deflated 53%)
  adding: qwen-1.5b-gsm8k-lora/checkpoint-468/adapter_config.json (deflated 60%)
  adding: qwen-1.5b-gsm8k-lora/checkpoint-468/chat_template.jinja (deflated 71%)
  adding: qwen-1.5b-gsm8k-lora/checkpoint-468/README.md (deflated 65%)
  adding: qwen-1.5b-gsm8k-lora/checkpoint-468/rng_state.pth (deflated 26%)
  adding: qwen-1.5b-gsm8k-lora/checkpoint-468/tokenizer.js

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>